In [5]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [6]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_final13_2.dat'
# 교차검증 횟수
cv_count = 10
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

### 데이터 준비
- 데이터를 읽어오고 필요한 전처리까지 다 한다음 입력데이터는 train_X, 결과데이터는 train_y라는 변수에 담아서 준비해주세요

In [7]:
# 데이터를 읽어온다.
train_df = pd.read_csv('E,notE(train).csv')
test_df = pd.read_csv('E,notE(test).csv')

display(train_df)
display(test_df)

,ID,기준년월,Segment,Life_Stage,연령,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,이용카드수_신용,...,이용건수_오프라인_R3M,이용후경과월_신용,_2순위업종_이용금액,이용개월수_신용_R6M,정상입금원금_B2M,정상입금원금_B5M,청구금액_R6M,청구금액_R3M,청구금액_B0,청구서발송여부_B0
0,TRAIN_000000,201807,D,자녀성장(2),40대,196,3681,196,1,1,...,17,0,1408,6,16125,9205,88693,46588,12226,1
1,TRAIN_000001,201807,E,자녀성장(1),30대,13475,13323,13475,1,1,...,51,0,2083,6,2420,2546,16861,10530,5834,1
2,TRAIN_000002,201807,C,자녀출산기,30대,23988,24493,23988,1,1,...,33,0,1539,6,14448,16949,165221,85931,21866,1
3,TRAIN_000003,201807,D,자녀성장(2),40대,3904,5933,3904,1,1,...,19,0,2284,6,13043,8418,127371,61518,16356,1
4,TRAIN_000004,201807,E,자녀성장(1),40대,1190,0,0,1,0,...,0,6,0,1,0,0,155,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2399995,TRAIN_399995,201812,E,노년생활,70대이상,10755,5640,7267,1,0,...,0,8,0,0,0,0,0,0,0,0
2399996,TRAIN_399996,201812,D,자녀성장(2),50대,27636,26357,27636,1,1,...,24,0,1810,6,10764,21831,99849,37515,14402,1
2399997,TRAIN_399997,201812,C,자녀출산기,30대,23187,17171,23187,1,1,...,32,1,2315,6,6106,3269,41073,22274,5731,1
2399998,TRAIN_399998,201812,E,자녀성장(1),40대,0,0,0,0,0,...,0,12,0,0,0,0,0,0,0,0


,ID,기준년월,Life_Stage,연령,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,...,이용건수_오프라인_R3M,이용후경과월_신용,_2순위업종_이용금액,이용개월수_신용_R6M,정상입금원금_B2M,정상입금원금_B5M,청구금액_R6M,청구금액_R3M,청구금액_B0,청구서발송여부_B0
0,TEST_00000,201807,자녀성장(1),40대,21458,13852,21458,2,2,51,...,48,0,2713,5,1172,1504,22151,11441,4931,1
1,TEST_00001,201807,자녀독립기,60대,18681,11065,10759,2,1,40,...,14,0,1321,6,4509,2763,32878,20522,10152,1
2,TEST_00002,201807,자녀성장(1),40대,40758,27071,40758,2,2,154,...,151,1,7271,6,11366,11267,71867,50508,13223,1
3,TEST_00003,201807,자녀성장(1),40대,5255,4827,5255,1,1,105,...,27,0,2043,5,875,0,4986,4604,2112,1
4,TEST_00004,201807,자녀성장(1),40대,16148,8011,14290,3,2,52,...,31,0,1364,6,1618,1347,10758,6788,4406,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,TEST_99995,201812,노년생활,60대,0,0,0,0,0,-2,...,0,12,0,0,0,0,0,0,0,0
599996,TEST_99996,201812,자녀출산기,30대,3110,1231,3110,1,1,4,...,13,0,0,6,252,302,2237,1256,359,1
599997,TEST_99997,201812,자녀성장(1),30대,0,0,0,0,0,6,...,0,12,0,0,0,0,0,0,0,0
599998,TEST_99998,201812,가족구축기,30대,173263,63592,113786,6,4,185,...,197,0,16296,6,2751,6215,108420,48141,21273,1


In [8]:
### Segment 변경
train_df['Segment'] = train_df['Segment'].apply(lambda x: 'E' if x == 'E' else 'Not_E')

In [14]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df

,ID,기준년월,Segment,Life_Stage,연령,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,이용카드수_신용,...,이용건수_오프라인_R3M,이용후경과월_신용,_2순위업종_이용금액,이용개월수_신용_R6M,정상입금원금_B2M,정상입금원금_B5M,청구금액_R6M,청구금액_R3M,청구금액_B0,청구서발송여부_B0
0,TRAIN_000000,201807,Not_E,자녀성장(2),40대,196,3681,196,1,1,...,17,0,1408,6,16125,9205,88693,46588,12226,1
1,TRAIN_000001,201807,E,자녀성장(1),30대,13475,13323,13475,1,1,...,51,0,2083,6,2420,2546,16861,10530,5834,1
2,TRAIN_000002,201807,Not_E,자녀출산기,30대,23988,24493,23988,1,1,...,33,0,1539,6,14448,16949,165221,85931,21866,1
3,TRAIN_000003,201807,Not_E,자녀성장(2),40대,3904,5933,3904,1,1,...,19,0,2284,6,13043,8418,127371,61518,16356,1
4,TRAIN_000004,201807,E,자녀성장(1),40대,1190,0,0,1,0,...,0,6,0,1,0,0,155,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,TEST_99995,201812,NaN,노년생활,60대,0,0,0,0,0,...,0,12,0,0,0,0,0,0,0,0
2999996,TEST_99996,201812,NaN,자녀출산기,30대,3110,1231,3110,1,1,...,13,0,0,6,252,302,2237,1256,359,1
2999997,TEST_99997,201812,NaN,자녀성장(1),30대,0,0,0,0,0,...,0,12,0,0,0,0,0,0,0,0
2999998,TEST_99998,201812,NaN,가족구축기,30대,173263,63592,113786,6,4,...,197,0,16296,6,2751,6215,108420,48141,21273,1


In [15]:
# 결과 데이터는 제거한다.
all_df.drop(columns=['Segment','ID','기준년월'], axis=1, inplace=True)
all_df

,Life_Stage,연령,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,이용카드수_신용,_1순위카드이용건수,이용금액_오프라인_B0M,정상청구원금_B5M,...,이용건수_오프라인_R3M,이용후경과월_신용,_2순위업종_이용금액,이용개월수_신용_R6M,정상입금원금_B2M,정상입금원금_B5M,청구금액_R6M,청구금액_R3M,청구금액_B0,청구서발송여부_B0
0,자녀성장(2),40대,196,3681,196,1,1,26,4043,14958,...,17,0,1408,6,16125,9205,88693,46588,12226,1
1,자녀성장(1),30대,13475,13323,13475,1,1,46,3980,3367,...,51,0,2083,6,2420,2546,16861,10530,5834,1
2,자녀출산기,30대,23988,24493,23988,1,1,28,4524,23963,...,33,0,1539,6,14448,16949,165221,85931,21866,1
3,자녀성장(2),40대,3904,5933,3904,1,1,1,3975,19614,...,19,0,2284,6,13043,8418,127371,61518,16356,1
4,자녀성장(1),40대,1190,0,0,1,0,-2,0,0,...,0,6,0,1,0,0,155,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2999995,노년생활,60대,0,0,0,0,0,-2,0,0,...,0,12,0,0,0,0,0,0,0,0
2999996,자녀출산기,30대,3110,1231,3110,1,1,4,407,992,...,13,0,0,6,252,302,2237,1256,359,1
2999997,자녀성장(1),30대,0,0,0,0,0,6,0,0,...,0,12,0,0,0,0,0,0,0,0
2999998,가족구축기,30대,173263,63592,113786,6,4,185,15388,27335,...,197,0,16296,6,2751,6215,108420,48141,21273,1


In [16]:
all_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 52 columns):
 #   Column              Dtype 
---  ------              ----- 
 0   Life_Stage          object
 1   연령                  object
 2   이용금액_R3M_신용체크       int64 
 3   _1순위카드이용금액          int64 
 4   이용금액_R3M_신용         int64 
 5   이용카드수_신용체크          int64 
 6   이용카드수_신용            int64 
 7   _1순위카드이용건수          int64 
 8   이용금액_오프라인_B0M       int64 
 9   정상청구원금_B5M          int64 
 10  정상청구원금_B0M          int64 
 11  정상청구원금_B2M          int64 
 12  이용개월수_신용_R12M       int64 
 13  연속유실적개월수_기본_24M_카드  int64 
 14  이용개월수_신판_R12M       int64 
 15  이용개월수_일시불_R12M      int64 
 16  이용금액_일시불_R6M        int64 
 17  이용금액_일시불_R3M        int64 
 18  이용금액_일시불_B0M        int64 
 19  이용건수_신용_R12M        int64 
 20  이용건수_신판_R12M        int64 
 21  이용금액_오프라인_R3M       int64 
 22  이용건수_일시불_R12M       int64 
 23  이용가맹점수              int64 
 24  이용금액_일시불_R12M       int64 
 25  이용개월수_전체_R6M      

In [17]:
# LabelEncoder 학습
Encoder1 = LabelEncoder()
Encoder2 = LabelEncoder()

Encoder1.fit(all_df['Life_Stage'])
Encoder2.fit(all_df['연령'])

LabelEncoder()

In [18]:
all_df['Life_Stage'] = Encoder1.transform(all_df['Life_Stage'])
all_df['연령'] = Encoder2.transform(all_df['연령'])

In [19]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

,copy,True
,with_mean,True
,with_std,True


In [20]:
train_df['Life_Stage'] = Encoder1.transform(train_df['Life_Stage'])
train_df['연령'] = Encoder2.transform(train_df['연령'])

In [23]:
# 입력과 결과로 나눈다.
X = train_df.drop(columns=['Segment','ID','기준년월'], axis=1)
y = train_df['Segment']

In [24]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[ 0.78248617, -0.11279485, -0.71919385, ...,  1.31055523,
         0.91338212,  0.54481299],
       [ 0.22559285, -0.92342879, -0.14848993, ..., -0.22006137,
         0.09862567,  0.54481299],
       [ 1.3393795 , -0.92342879,  0.30333704, ...,  2.98061593,
         2.14214499,  0.54481299],
       ...,
       [ 1.3393795 , -0.92342879,  0.26891172, ...,  0.27845661,
         0.08549677,  0.54481299],
       [ 0.22559285, -0.11279485, -0.72761753, ..., -0.66704658,
        -0.6450053 , -1.83549223],
       [ 0.22559285, -0.11279485,  0.19481777, ..., -0.25512403,
        -0.20881998,  0.54481299]])

In [25]:
scaler_columns = X.columns.tolist()
scaler_columns

['Life_Stage',
 '연령',
 '이용금액_R3M_신용체크',
 '_1순위카드이용금액',
 '이용금액_R3M_신용',
 '이용카드수_신용체크',
 '이용카드수_신용',
 '_1순위카드이용건수',
 '이용금액_오프라인_B0M',
 '정상청구원금_B5M',
 '정상청구원금_B0M',
 '정상청구원금_B2M',
 '이용개월수_신용_R12M',
 '연속유실적개월수_기본_24M_카드',
 '이용개월수_신판_R12M',
 '이용개월수_일시불_R12M',
 '이용금액_일시불_R6M',
 '이용금액_일시불_R3M',
 '이용금액_일시불_B0M',
 '이용건수_신용_R12M',
 '이용건수_신판_R12M',
 '이용금액_오프라인_R3M',
 '이용건수_일시불_R12M',
 '이용가맹점수',
 '이용금액_일시불_R12M',
 '이용개월수_전체_R6M',
 '이용금액_오프라인_R6M',
 '이용건수_오프라인_B0M',
 '이용건수_신용_R6M',
 '이용건수_신용_B0M',
 '_2순위쇼핑업종_이용금액',
 '이용건수_신판_R6M',
 '이용건수_신용_R3M',
 '이용건수_신판_B0M',
 '이용건수_일시불_R6M',
 '이용건수_신판_R3M',
 '이용건수_일시불_B0M',
 '이용건수_일시불_R3M',
 '_3순위쇼핑업종_이용금액',
 '정상입금원금_B0M',
 '_3순위업종_이용금액',
 '이용개월수_전체_R3M',
 '이용건수_오프라인_R3M',
 '이용후경과월_신용',
 '_2순위업종_이용금액',
 '이용개월수_신용_R6M',
 '정상입금원금_B2M',
 '정상입금원금_B5M',
 '청구금액_R6M',
 '청구금액_R3M',
 '청구금액_B0',
 '청구서발송여부_B0']

In [26]:
train_X = X2
train_y = y

In [27]:
le = LabelEncoder()
train_y = le.fit_transform(train_y)

### 기본 모델 사용하기
- 기본 모델 중에 만족하는 것을 찾았다면 하이퍼 파라미터 튜닝 과정은 생략하세요

In [28]:
model5 = LGBMClassifier(device='cpu', objective='multiclass', num_class=5, verbose=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
r1 = cross_val_score(model5, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r1.mean()}')

f1_score_list.append(r1.mean())
model_name_list.append("LGBMClassifier")

평균 f1 Score : 0.91715375


In [19]:
# CPU 기반 XGBoost 모델
model6 = XGBClassifier(
    n_jobs=-1,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# 교차 검증
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

# f1_weighted 사용
r2 = cross_val_score(model6, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r2.mean():.4f}')

f1_score_list.append(r2.mean())
model_name_list.append("XGBClassifier")

평균 f1 Score : 0.9074


In [20]:
df = pd.DataFrame({
    'Model': model_name_list,
    'f1 score': f1_score_list
})
df = df.dropna()

In [21]:
df

,Model,f1 score
0,LGBMClassifier,0.906030
1,XGBClassifier,0.907386


In [22]:
final_model=model6.fit(train_X, train_y)

In [27]:
with open(best_model_path, 'wb') as fp:
    pickle.dump(model6, fp)
    pickle.dump(scalerX, fp)
    pickle.dump(scaler_columns, fp)
    pickle.dump(Encoder1, fp)
    pickle.dump(Encoder2, fp)
    pickle.dump(Encoder3, fp)
    pickle.dump(Encoder4, fp)
    pickle.dump(Encoder5, fp)
    pickle.dump(Encoder6, fp)
    pickle.dump(le, fp)  # ← LabelEncoder 객체 추가

print('저장완료')

저장완료
